In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score, classification_report
import joblib

In [2]:
train_df = pd.read_csv("data/twitter_training.csv")
val_df = pd.read_csv("data/twitter_validation.csv")
train_df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [3]:
val_df.head()

,3364,Facebook,Irrelevant,"I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣"
0,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
1,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
2,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
3,4433,Google,Neutral,Now the President is slapping Americans in the...
4,6273,FIFA,Negative,Hi @EAHelp I’ve had Madeleine McCann in my cel...


In [4]:
print("train shape",train_df.shape)
print("validation shape",val_df.shape)

train shape (74681, 4)
validation shape (999, 4)


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [6]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 4 columns):
 #   Column                                                                                                                                                                                                                                              Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                                              --------------  ----- 
 0   3364                                                                                                                                                                                                                                                999 non-null    int64 
 1   Facebook                                                                   

In [7]:
train_df.isnull().sum()

2401                                                       0
Borderlands                                                0
Positive                                                   0
im getting on borderlands and i will murder you all ,    686
dtype: int64

In [8]:
val_df.isnull().sum()

3364                                                                                                                                                                                                                                                  0
Facebook                                                                                                                                                                                                                                              0
Irrelevant                                                                                                                                                                                                                                            0
I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣    0
dtype: i

In [9]:
train_df.iloc[:, 2].value_counts()

Positive
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [10]:
val_df.iloc[:, 2].value_counts()

Irrelevant
Neutral       285
Positive      277
Negative      266
Irrelevant    171
Name: count, dtype: int64

In [11]:
def preprocess_data(df):
    df = df.copy()
    df.columns = ["id", "entity", "sentiment", "text"]
    df = df[["text", "sentiment"]]
    df = df.dropna()
    df = df[df["sentiment"] != "Irrelevant"]
    return df

In [12]:
train_data = preprocess_data(train_df)
train_data.head()

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive


In [13]:
train_data["sentiment"].value_counts()

sentiment
Negative    22358
Positive    20654
Neutral     18108
Name: count, dtype: int64

In [14]:
val_data = preprocess_data(val_df)
val_data.head()

,text,sentiment
0,BBC News - Amazon boss Jeff Bezos rejects clai...,Neutral
1,@Microsoft Why do I pay for WORD when it funct...,Negative
2,"CSGO matchmaking is so full of closet hacking,...",Negative
3,Now the President is slapping Americans in the...,Neutral
4,Hi @EAHelp I’ve had Madeleine McCann in my cel...,Negative


In [15]:
def clean_text(text):
    text = text.lower()  # lowercase
    
    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # remove mentions and hashtags
    text = re.sub(r'@\w+|#', '', text)
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

train_data["text"] = train_data["text"].apply(clean_text)
val_data["text"] = val_data["text"].apply(clean_text)

In [16]:
train_data.head()

,text,sentiment
0,i am coming to the borders and i will kill you...,Positive
1,im getting on borderlands and i will kill you all,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive


In [17]:
val_data.head()

,text,sentiment
0,bbc news amazon boss jeff bezos rejects claims...,Neutral
1,why do i pay for word when it functions so poo...,Negative
2,csgo matchmaking is so full of closet hacking ...,Negative
3,now the president is slapping americans in the...,Neutral
4,hi i’ve had madeleine mccann in my cellar for ...,Negative


In [18]:
X_train = train_data["text"]
y_train = train_data["sentiment"]

X_val = val_data["text"]
y_val = val_data["sentiment"]

In [19]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

In [20]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)
lr_pred = lr_model.predict(X_val_vec)

svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)
svm_pred = svm_model.predict(X_val_vec)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_val, lr_pred))
print(classification_report(y_val, lr_pred))
print("=== SVM ===")
print("Accuracy:", accuracy_score(y_val, svm_pred))
print(classification_report(y_val, svm_pred))

=== Logistic Regression ===
Accuracy: 0.8671497584541062
              precision    recall  f1-score   support

    Negative       0.82      0.91      0.86       266
     Neutral       0.91      0.80      0.85       285
    Positive       0.88      0.90      0.89       277

    accuracy                           0.87       828
   macro avg       0.87      0.87      0.87       828
weighted avg       0.87      0.87      0.87       828

=== SVM ===
Accuracy: 0.8792270531400966
              precision    recall  f1-score   support

    Negative       0.85      0.93      0.89       266
     Neutral       0.92      0.82      0.87       285
    Positive       0.88      0.89      0.88       277

    accuracy                           0.88       828
   macro avg       0.88      0.88      0.88       828
weighted avg       0.88      0.88      0.88       828



In [21]:
lr_score = f1_score(y_val, lr_pred, average='weighted')
svm_score = f1_score(y_val, svm_pred, average='weighted')
print("LR F1:", lr_score)
print("SVM F1:", svm_score)

LR F1: 0.8669525814053286
SVM F1: 0.8789651619829274


In [22]:
best_model = svm_model if svm_score > lr_score else lr_model
print("Best model:", "SVM" if svm_score > lr_score else "Logistic Regression")
print(best_model)

Best model: SVM
LinearSVC()


In [25]:
joblib.dump(best_model, "models/best_model.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")

['models/vectorizer.pkl']

In [24]:
text = ["that's enough!"]
vec = vectorizer.transform(text)
best_model.predict(vec)

array(['Positive'], dtype=object)